In [ ]:
# Traffic Application

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [3]:
spark = SparkSession.builder \
    .appName("CSV Reader") \
    .master("local[*]") \
    .getOrCreate()

csv_file_path_utd = "/Users/shaydabanihashemi/data/lake/bronze/utd19_u.csv"

metrics = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv(csv_file_path_utd)
metrics = metrics.filter(metrics.error.isNull())
metrics = metrics.filter(~metrics.speed.isNull())

csv_file_path_detectors = "/Users/shaydabanihashemi/data/lake/bronze/detectors_public.csv"

detectors = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv(csv_file_path_detectors)

metrics = metrics.withColumnRenamed('detid', 'detid_metrics')
detectors = detectors.withColumnRenamed('detid', 'detid_detectors')

df = metrics.join(detectors, metrics.city == detectors.citycode, how='inner')

#df.write.mode("overwrite").parquet('/Users/shaydabanihashemi/data/lake/silver/traffic.parquet')

In [4]:
metrics.select('city').distinct().show()

+----------+
|      city|
+----------+
|    bolton|
|birmingham|
| constance|
| groningen|
| innsbruck|
|manchester|
| rotterdam|
|    torino|
+----------+



In [5]:
detectors.select('citycode').distinct().show()

+-----------+
|   citycode|
+-----------+
|   augsburg|
|      tokyo|
|   bordeaux|
|  innsbruck|
|     kassel|
| strasbourg|
|      paris|
|    toronto|
|  melbourne|
|  constance|
|  darmstadt|
|  groningen|
|losanageles|
|     luzern|
|      basel|
|     bolton|
|     london|
|    hamburg|
|       bern|
|  rotterdam|
+-----------+
only showing top 20 rows



In [6]:
df.select('city').distinct().show()
df.select('citycode').distinct().show()

+----------+
|      city|
+----------+
|    bolton|
|birmingham|
| constance|
| groningen|
| innsbruck|
|manchester|
| rotterdam|
|    torino|
+----------+



+----------+
|  citycode|
+----------+
|    bolton|
|birmingham|
| constance|
| groningen|
| innsbruck|
|manchester|
| rotterdam|
|    torino|
+----------+



In [7]:
citycodes = df.select("citycode").distinct().collect()
basepath = "/Users/shaydabanihashemi/data/lake/silver/"
citycode_values = [row['citycode'] for row in citycodes]
for city in citycode_values:
    file_path = (f"{basepath}{city}.parquet")
    df.filter(df["citycode"] == city).write.mode("overwrite").parquet(file_path)

In [8]:
df.show(30)

+----------+--------+-------------+----+----+-----+----------+-----+---------------+-----------------+-----------------+----------+--------------------+-----+----------+-----+------+-----------------+----------------+
|       day|interval|detid_metrics|flow| occ|error|      city|speed|detid_detectors|           length|              pos|    fclass|                road|limit|  citycode|lanes|linkid|             long|             lat|
+----------+--------+-------------+----+----+-----+----------+-----+---------------+-----------------+-----------------+----------+--------------------+-----+----------+-----+------+-----------------+----------------+
|2017-10-24|       0|      N11131D|12.0|NULL| NULL|birmingham| 80.0|        N53161K|0.248647678433248| 0.22682280340806| secondary|   Constitution Hill|   48|birmingham|    1|    66|-1.90429855649954|52.4876077517489|
|2017-10-24|       0|      N11131D|12.0|NULL| NULL|birmingham| 80.0|        N51131R|0.275503362502858| 0.27394892804737| seconda

In [9]:
df.count()

968495907

In [10]:
max_city_flows = df.groupby("citycode").max("flow").persist()
max_city_flows = max_city_flows.withColumnRenamed('max(flow)', 'max_city_flows')
max_city_flows.show(300)
max_city_flows.count()

+----------+--------------+
|  citycode|max_city_flows|
+----------+--------------+
| innsbruck|     3482.4001|
| constance|        1281.6|
| groningen|        1245.0|
|    bolton|        2364.0|
| rotterdam|        2352.0|
|manchester|        2940.0|
|    torino|        4284.0|
|birmingham|        3588.0|
+----------+--------------+



8

In [11]:
joined_max_city_flows = max_city_flows.join(df, on=[(df.citycode == max_city_flows.citycode) & (df.flow == max_city_flows.max_city_flows)], how='inner')
joined_max_city_flows.show(150)
joined_max_city_flows.filter(col("city") == "birmingham").count()


+----------+--------------+----------+--------+-------------+------+----------------+-----+----------+-----+---------------+-----------------+-----------------+----------+--------------------+-----+----------+-----+------+-----------------+----------------+
|  citycode|max_city_flows|       day|interval|detid_metrics|  flow|             occ|error|      city|speed|detid_detectors|           length|              pos|    fclass|                road|limit|  citycode|lanes|linkid|             long|             lat|
+----------+--------------+----------+--------+-------------+------+----------------+-----+----------+-----+---------------+-----------------+-----------------+----------+--------------------+-----+----------+-----+------+-----------------+----------------+
|birmingham|        3588.0|2017-11-15|   35400|      N33132Z|3588.0|            NULL| NULL|birmingham| 17.0|        N53161K|0.248647678433248| 0.22682280340806| secondary|   Constitution Hill|   48|birmingham|    1|    66|-1.9

66

In [12]:
joined_max_city_flows.count()

1264

In [13]:
joined_max_city_flows.filter(col("city") == "birmingham").count()

66

In [14]:
city_code_list = df.select("road").where(df.citycode == 'birmingham').distinct()
city_code_list.show(truncate=False)

+--------------------+
|road                |
+--------------------+
|Digbeth             |
|James Watt Queensway|
|Camp Hill           |
|Bristol Road        |
|Jennens Road        |
|Lee Bank Middleway  |
|Constitution Hill   |
|Queensway           |
|Rea Street          |
|Moat Lane           |
|Smallbrook Queensway|
|Hockley Street      |
|Bradford Street     |
|Gooch Street        |
|Great Hampton Street|
|Alcester Street     |
|Bristol Street      |
|Icknield Street     |
|Pershore Street     |
|Masshouse Lane      |
+--------------------+
only showing top 20 rows



In [15]:
from pyspark.sql.functions import max
df_4 = df.select(max(df.flow))
df_4.show()

+---------+
|max(flow)|
+---------+
|   4284.0|
+---------+

